# Theorem 7 — local Laplace modal stability

**Formal source:** [`../07_laplace_modal_stability.md`](../07_laplace_modal_stability.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rho, omega = 0.4, 5.0
for time in [0, 0.13, 0.7, 2.0]:
    rotation = np.array([[math.cos(omega * time), -math.sin(omega * time)], [math.sin(omega * time), math.cos(omega * time)]])
    transition = math.exp(-rho * time) * rotation
    np.testing.assert_allclose(np.linalg.norm(transition, 2), math.exp(-rho * time), rtol=1e-12, atol=1e-12)
times = np.array([0, 0.03, 0.11, 0.5, 0.93])
trajectory = np.exp(-rho * times) * (np.cos(omega * times) + 0.3 * np.sin(omega * times))
assert np.all(np.isfinite(trajectory))
print({"transition_norm_t2": float(math.exp(-0.8)), "irregular_query": trajectory.tolist()})

In [ ]:
print('THEORY_DEMO_PASS::07_laplace_modal_stability')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')